In [1]:
import os
import sys
import shutil

# 切换到工作目录
WORK_DIR = "/kaggle/working"
os.chdir(WORK_DIR)

# 克隆官方MVSS-Net仓库
if not os.path.exists("MVSS-Net"):
    !git clone https://github.com/dong03/MVSS-Net.git

# 将官方代码加入系统路径
sys.path.append(os.path.join(WORK_DIR, "MVSS-Net"))

# 安装依赖（官方requirements + 补充依赖）
!pip install -q opencv-python scikit-image tqdm matplotlib albumentations

# 基础库导入
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# 检查GPU
print("GPU可用:", torch.cuda.is_available())
print("GPU数量:", torch.cuda.device_count())

Cloning into 'MVSS-Net'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 86 (delta 0), reused 1 (delta 0), pack-reused 83 (from 1)
Receiving objects: 100% (86/86), 6.17 MiB | 24.39 MiB/s, done.
Resolving deltas: 100% (35/35), done.
GPU可用: True
GPU数量: 2


In [2]:
# ====================== 配置项 ======================
DATASET_ROOT = "/kaggle/input/datasets/chongtrung/casia-v2/CASIA2"  # 替换为你的数据集路径
INDEX_SAVE_PATH = "/kaggle/working/casiav2_train.txt"
# ====================================================

# 遍历数据集生成索引文件
authentic_dir = os.path.join(DATASET_ROOT, "Au")  # 真实图像目录
tampered_dir = os.path.join(DATASET_ROOT, "Tp")   # 篡改图像目录
mask_dir = os.path.join(DATASET_ROOT, "Groundtruth") # 标注mask目录

with open(INDEX_SAVE_PATH, "w") as f:
    # 写入真实图像
    for img_name in os.listdir(authentic_dir):
        if img_name.lower().endswith(('.jpg', '.png', '.bmp')):
            img_path = os.path.join(authentic_dir, img_name)
            f.write(f"{img_path} None 0\n")
    
    # 写入篡改图像与对应mask
    for img_name in os.listdir(tampered_dir):
        if img_name.lower().endswith(('.jpg', '.png', '.bmp')):
            img_path = os.path.join(tampered_dir, img_name)
            # 按CASIA命名规则匹配mask（可根据你的数据集调整）
            mask_name = img_name.replace(".jpg", "_gt.png").replace(".bmp", "_gt.png")
            mask_path = os.path.join(mask_dir, mask_name)
            if os.path.exists(mask_path):
                f.write(f"{img_path} {mask_path} 1\n")

print(f"索引文件生成完成，保存至: {INDEX_SAVE_PATH}")
print("可通过head命令检查格式:")
!head -5 /kaggle/working/casiav2_train.txt

索引文件生成完成，保存至: /kaggle/working/casiav2_train.txt
可通过head命令检查格式:
/kaggle/input/datasets/chongtrung/casia-v2/CASIA2/Au/Au_pla_30322.jpg None 0
/kaggle/input/datasets/chongtrung/casia-v2/CASIA2/Au/Au_ani_10197.jpg None 0
/kaggle/input/datasets/chongtrung/casia-v2/CASIA2/Au/Au_nat_00098.jpg None 0
/kaggle/input/datasets/chongtrung/casia-v2/CASIA2/Au/Au_nat_30407.jpg None 0
/kaggle/input/datasets/chongtrung/casia-v2/CASIA2/Au/Au_arc_20034.jpg None 0


In [3]:
!ls ../input/datasets/chongtrung/

casia-v2


MVSS-Net 训练需要边缘标注作为辅助监督（Edge Supervision），这里补充基于 Canny 算子的边缘 mask 批量生成函数，完全对齐论文中的边缘监督设置

In [4]:
def generate_edge_mask(mask_path, output_path=None, low_thresh=30, high_thresh=100):
    """
    从二值篡改mask生成边缘标注mask
    Args:
        mask_path: 输入二值GT mask路径
        output_path: 保存边缘mask的路径，为None则返回numpy数组
        low_thresh, high_thresh: Canny双阈值
    Returns:
        edge_mask: 单通道二值边缘图
    """
    # 读取GT mask并转灰度
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None
    
    # 二值化确保mask为0/255
    _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    
    # Canny提取边缘
    edge = cv2.Canny(mask, low_thresh, high_thresh)
    
    # 膨胀1像素，增强边缘监督信号
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3,3))
    edge = cv2.dilate(edge, kernel, iterations=1)
    
    if output_path:
        cv2.imwrite(output_path, edge)
    return edge

# ====================== 批量生成边缘mask ======================
EDGE_SAVE_DIR = "/kaggle/working/edge_masks"
os.makedirs(EDGE_SAVE_DIR, exist_ok=True)

# 读取索引文件，为所有篡改图像生成边缘mask
edge_index_lines = []
with open(INDEX_SAVE_PATH, "r") as f:
    for line in tqdm(f.readlines(), desc="生成边缘mask"):
        img_path, mask_path, label = line.strip().split()
        label = int(label)
        
        if label == 1 and mask_path != "None":
            # 生成边缘mask
            edge_name = os.path.basename(mask_path).replace("_gt.png", "_edge.png")
            edge_save_path = os.path.join(EDGE_SAVE_DIR, edge_name)
            generate_edge_mask(mask_path, edge_save_path)
            edge_index_lines.append(f"{img_path} {mask_path} {edge_save_path} {label}\n")
        else:
            # 真实图像边缘mask填None
            edge_index_lines.append(f"{img_path} {mask_path} None {label}\n")

# 保存带边缘路径的完整索引
EDGE_INDEX_PATH = "/kaggle/working/casiav2_train_with_edge.txt"
with open(EDGE_INDEX_PATH, "w") as f:
    f.writelines(edge_index_lines)

print(f"边缘mask生成完成，保存在: {EDGE_SAVE_DIR}")
print(f"完整索引文件: {EDGE_INDEX_PATH}")

生成边缘mask: 100%|██████████| 9495/9495 [00:18<00:00, 510.03it/s]  

边缘mask生成完成，保存在: /kaggle/working/edge_masks
完整索引文件: /kaggle/working/casiav2_train_with_edge.txt


In [5]:
!head -5 /kaggle/working/casiav2_train_with_edge.txt

/kaggle/input/datasets/chongtrung/casia-v2/CASIA2/Au/Au_pla_30322.jpg None None 0
/kaggle/input/datasets/chongtrung/casia-v2/CASIA2/Au/Au_ani_10197.jpg None None 0
/kaggle/input/datasets/chongtrung/casia-v2/CASIA2/Au/Au_nat_00098.jpg None None 0
/kaggle/input/datasets/chongtrung/casia-v2/CASIA2/Au/Au_nat_30407.jpg None None 0
/kaggle/input/datasets/chongtrung/casia-v2/CASIA2/Au/Au_arc_20034.jpg None None 0


In [6]:
class MVSSDataset(Dataset):
    """
    MVSS-Net 自定义数据集，对齐官方索引格式
    索引文件每行格式: img_path mask_path edge_mask_path label
    """
    def __init__(self, index_file, img_size=512, is_train=True):
        self.is_train = is_train
        self.img_size = img_size
        self.samples = []
        
        # 读取索引文件
        with open(index_file, "r") as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) == 4:
                    img_path, mask_path, edge_path, label = parts
                else:
                    img_path, mask_path, label = parts
                    edge_path = "None"
                self.samples.append({
                    "img_path": img_path,
                    "mask_path": mask_path,
                    "edge_path": edge_path,
                    "label": int(label)
                })
        
        # 数据增强与预处理
        if is_train:
            self.transform = A.Compose([
                A.Resize(height=img_size, width=img_size),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.Normalize(mean=[0.485, 0.456, 0.406], 
                            std=[0.229, 0.224, 0.225]),
                ToTensorV2(),
            ], additional_targets={"mask": "mask", "edge_mask": "mask"})
        else:
            self.transform = A.Compose([
                A.Resize(height=img_size, width=img_size),
                A.Normalize(mean=[0.485, 0.456, 0.406], 
                            std=[0.229, 0.224, 0.225]),
                ToTensorV2(),
            ], additional_targets={"mask": "mask", "edge_mask": "mask"})
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # 读取RGB图像
        image = cv2.imread(sample["img_path"])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w = image.shape[:2]  # 记录原图尺寸
        
        # 读取GT mask（真实图像为全黑）
        if sample["mask_path"] != "None":
            mask = cv2.imread(sample["mask_path"], cv2.IMREAD_GRAYSCALE)
            # 强制对齐到原图尺寸
            if mask.shape[:2] != (h, w):
                mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
            mask = (mask > 127).astype(np.float32)
        else:
            mask = np.zeros((h, w), dtype=np.float32)
        
        # 读取边缘mask（真实图像为全黑）
        if sample["edge_path"] != "None":
            edge_mask = cv2.imread(sample["edge_path"], cv2.IMREAD_GRAYSCALE)
            # 强制对齐到原图尺寸
            if edge_mask.shape[:2] != (h, w):
                edge_mask = cv2.resize(edge_mask, (w, h), interpolation=cv2.INTER_NEAREST)
            edge_mask = (edge_mask > 127).astype(np.float32)
        else:
            edge_mask = np.zeros((h, w), dtype=np.float32)
        
        # 应用预处理
        transformed = self.transform(image=image, mask=mask, edge_mask=edge_mask)
        image = transformed["image"]
        mask = transformed["mask"].unsqueeze(0)  # [1, H, W]
        edge_mask = transformed["edge_mask"].unsqueeze(0)  # [1, H, W]
        
        label = torch.tensor(sample["label"], dtype=torch.float32)
        
        return {
            "image": image,
            "mask": mask,
            "edge_mask": edge_mask,
            "label": label
        }

# ====================== 测试Dataset ======================
test_dataset = MVSSDataset(EDGE_INDEX_PATH, img_size=512, is_train=False)
sample = test_dataset[0]
print("图像shape:", sample["image"].shape)
print("Mask shape:", sample["mask"].shape)
print("边缘Mask shape:", sample["edge_mask"].shape)
print("图像标签:", sample["label"].item())

图像shape: torch.Size([3, 512, 512])
Mask shape: torch.Size([1, 512, 512])
边缘Mask shape: torch.Size([1, 512, 512])
图像标签: 0.0


官方未提供： 完整的三尺度加权损失函数，完全复现论文公式：
\(L_{total} = \alpha \cdot L_{seg} + \beta \cdot L_{edge} + (1-\alpha-\beta) \cdot L_{clf}\)
其中论文默认 \(\alpha=0.16, \beta=0.04\)

In [7]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, pred, target):
        # 自动把target对齐到pred的空间尺寸
        if pred.shape[2:] != target.shape[2:]:
            target = nn.functional.interpolate(target, size=pred.shape[2:], mode='nearest')
        
        pred = torch.sigmoid(pred)
        pred_flat = pred.view(-1)
        target_flat = target.view(-1)
        
        intersection = (pred_flat * target_flat).sum()
        dice = (2. * intersection + self.smooth) / (pred_flat.sum() + target_flat.sum() + self.smooth)
        return 1 - dice


class MVSSLoss(nn.Module):
    def __init__(self, alpha=0.7, beta=0.1):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.seg_loss = DiceLoss()
        self.edge_loss = DiceLoss()
        self.clf_loss = nn.BCEWithLogitsLoss()
    
    def forward(self, seg_pred, edge_pred, clf_pred, mask_gt, edge_gt, label_gt):
        loss_seg = self.seg_loss(seg_pred, mask_gt)
        
        edge_gt_down = nn.functional.interpolate(edge_gt, size=edge_pred.shape[2:], mode='nearest')
        loss_edge = self.edge_loss(edge_pred, edge_gt_down)
        
        # 关键修复：只压缩第1维（通道维），保证输出始终是 [B]
        loss_clf = self.clf_loss(clf_pred.squeeze(1), label_gt)
        
        total_loss = self.alpha * loss_seg + self.beta * loss_edge + (1 - self.alpha - self.beta) * loss_clf
        
        return total_loss, {
            "loss_seg": loss_seg.item(),
            "loss_edge": loss_edge.item(),
            "loss_clf": loss_clf.item(),
            "loss_total": total_loss.item()
        }
        

class BCELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.loss = nn.BCEWithLogitsLoss()
    
    def forward(self, pred, target):
        return self.loss(pred, target)


# 测试损失
criterion = MVSSLoss()
print("损失函数初始化完成")

损失函数初始化完成


In [8]:
# 导入官方模型
from models.mvssnet import MVSSNet

# 初始化模型
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MVSSNet(n_input=3, nclass=1).to(device)

# 多GPU并行（Kaggle T4 x2）
# if torch.cuda.device_count() > 1:
#     model = nn.DataParallel(model)
#     print("启用多GPU并行训练")

# 统计参数量
total_params = sum(p.numel() for p in model.parameters())
print(f"模型参数量: {total_params / 1e6:.2f} M")

Downloading: "https://download.pytorch.org/models/resnet50-19c8e357.pth" to /root/.cache/torch/hub/checkpoints/resnet50-19c8e357.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 273MB/s]


load pretrain success
模型参数量: 49.49 M


In [9]:
# 补全缺失的索引文件路径变量
EDGE_INDEX_PATH = "/kaggle/working/casiav2_train_with_edge.txt"

# ====================== 超参数配置 ======================
BATCH_SIZE = 4       # T4 16G显存可设8-16
IMG_SIZE = 512
EPOCHS = 30
LR = 1e-4
WEIGHT_DECAY = 1e-5
NUM_WORKERS = 2
SAVE_DIR = "/kaggle/working/checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

# ====================== 划分训练验证集 ======================
# 简单按9:1划分，也可自定义验证集索引
all_samples = []
with open(EDGE_INDEX_PATH, "r") as f:
    all_samples = f.readlines()

split_idx = int(len(all_samples) * 0.9)
train_lines = all_samples[:split_idx]
val_lines = all_samples[split_idx:]

# 保存临时索引
train_idx_path = "/kaggle/working/train_tmp.txt"
val_idx_path = "/kaggle/working/val_tmp.txt"
with open(train_idx_path, "w") as f: f.writelines(train_lines)
with open(val_idx_path, "w") as f: f.writelines(val_lines)

# 构建Dataset与DataLoader
train_dataset = MVSSDataset(train_idx_path, img_size=IMG_SIZE, is_train=True)
val_dataset = MVSSDataset(val_idx_path, img_size=IMG_SIZE, is_train=False)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, 
    shuffle=True, num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, 
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)

# 优化器与学习率调度
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

print(f"训练集样本数: {len(train_dataset)}")
print(f"验证集样本数: {len(val_dataset)}")
print(f"每轮迭代步数: {len(train_loader)}")

训练集样本数: 8545
验证集样本数: 950
每轮迭代步数: 2137


In [10]:
# 在训练配置处初始化缩放器
scaler = torch.cuda.amp.GradScaler()

def train_one_epoch(model, loader, criterion, optimizer, device, scaler):
    model.train()
    loss_records = {"loss_seg":[], "loss_edge":[], "loss_clf":[], "loss_total":[]}
    
    pbar = tqdm(loader, desc="Training")
    for batch in pbar:
        images = batch["image"].to(device)
        masks = batch["mask"].to(device)
        edges = batch["edge_mask"].to(device)
        labels = batch["label"].to(device)
        
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast():
            # 模型输出顺序：边缘预测(128x128)、主分割预测(512x512)
            edge_pred, seg_pred = model(images)
            # 生成图像级分类预测
            clf_pred = seg_pred.mean(dim=[2, 3])
            
            # 损失计算：严格6个参数，顺序一一对应
            total_loss, loss_dict = criterion(
                seg_pred,   # 1. 主分割预测 → seg_pred
                edge_pred,  # 2. 边缘分支预测 → edge_pred
                clf_pred,   # 3. 分类预测 → clf_pred
                masks,      # 4. 分割GT标注 → mask_gt
                edges,      # 5. 边缘GT标注 → edge_gt （必填）
                labels      # 6. 图像级标签 → label_gt （必填）
            )
        
        scaler.scale(total_loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        for k, v in loss_dict.items():
            loss_records[k].append(v)
        
        pbar.set_postfix({
            "loss": f"{loss_dict['loss_total']:.4f}",
            "seg": f"{loss_dict['loss_seg']:.4f}"
        })
    
    avg_loss = {k: np.mean(v) for k, v in loss_records.items()}
    return avg_loss

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    loss_records = {"loss_seg":[], "loss_edge":[], "loss_clf":[], "loss_total":[]}
    
    for batch in tqdm(loader, desc="Validating"):
        images = batch["image"].to(device)
        masks = batch["mask"].to(device)
        edges = batch["edge_mask"].to(device)
        labels = batch["label"].to(device)
        
        edge_pred, seg_pred = model(images)
        clf_pred = seg_pred.mean(dim=[2, 3])
        
        # 验证也必须传全6个参数，和训练保持一致
        total_loss, loss_dict = criterion(
            seg_pred, edge_pred, clf_pred, masks, edges, labels
        )
        
        for k, v in loss_dict.items():
            loss_records[k].append(v)
    
    avg_loss = {k: np.mean(v) for k, v in loss_records.items()}
    return avg_loss

# ====================== 开始训练 ======================
best_val_loss = float("inf")
criterion = MVSSLoss().to(device)

for epoch in range(1, EPOCHS + 1):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch}/{EPOCHS}")
    print(f"当前学习率: {optimizer.param_groups[0]['lr']:.6f}")
    
    # 训练
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler)
    print(f"训练损失: total={train_loss['loss_total']:.4f} | seg={train_loss['loss_seg']:.4f} | edge={train_loss['loss_edge']:.4f} | clf={train_loss['loss_clf']:.4f}")
    
    # 验证
    val_loss = validate(model, val_loader, criterion, device)
    print(f"验证损失: total={val_loss['loss_total']:.4f} | seg={val_loss['loss_seg']:.4f} | edge={val_loss['loss_edge']:.4f} | clf={val_loss['loss_clf']:.4f}")
    
    # 学习率更新
    scheduler.step()
    
    # 保存最佳模型
    if val_loss["loss_total"] < best_val_loss:
        best_val_loss = val_loss["loss_total"]
        save_path = os.path.join(SAVE_DIR, "mvssnet_best.pth")
        torch.save(model.state_dict(), save_path)
        print(f"✅ 最佳模型已保存，验证损失: {best_val_loss:.4f}")
    
    # 保存最新模型
    torch.save(model.state_dict(), os.path.join(SAVE_DIR, "mvssnet_latest.pth"))

print("\n🎉 训练完成！模型权重保存在 /kaggle/working/checkpoints/")


Epoch 1/30
当前学习率: 0.000100


Training: 100%|██████████| 2137/2137 [05:46<00:00,  6.17it/s, loss=0.8026, seg=1.0000]


训练损失: total=0.8311 | seg=0.9566 | edge=0.9970 | clf=0.3093


Validating: 100%|██████████| 238/238 [00:33<00:00,  7.21it/s]


验证损失: total=0.9039 | seg=0.7855 | edge=0.9764 | clf=1.2820
✅ 最佳模型已保存，验证损失: 0.9039

Epoch 2/30
当前学习率: 0.000100


...

验证损失: total=nan | seg=nan | edge=0.9539 | clf=nan

Epoch 30/30
当前学习率: 0.000001


Training: 100%|██████████| 2137/2137 [05:28<00:00,  6.51it/s, loss=0.8000, seg=1.0000]


训练损失: total=0.7067 | seg=0.8256 | edge=0.9845 | clf=0.1516


Validating: 100%|██████████| 238/238 [00:31<00:00,  7.53it/s]


验证损失: total=nan | seg=nan | edge=0.9548 | clf=nan

🎉 训练完成！模型权重保存在 /kaggle/working/checkpoints/
